# Quantvesting

## Premium Investment Terminal — Phase 2

A customer-first executive view over the existing Quantvesting engine. The terminal prioritises five value KPIs, portfolio health and concentration; deeper Quantvesting metrics remain available in drill-down views. **No investment calculations live in the notebook.**

In [1]:
!pip install -q ta pyxirr matplotlib plotly ipywidgets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.4/532.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.2 MB/s eta 0:00:00


### 1. Environment & controls

Enter the `PORTFOLIO_ID` when prompted; do not edit notebook code for another portfolio. Keep `EOD_RUN=False` unless this is the final end-of-day snapshot.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import sys
PROJECT = "/content/drive/My Drive/quantvesting_v3"
MARKET_DATA_DIR = PROJECT + "/market_data"
PORTFOLIO_ID = input("Enter Portfolio ID (default: ankit): ").strip() or "ankit"
PORTFOLIO_DATA_DIR = PROJECT + f"/portfolio_data/{PORTFOLIO_ID}"
EOD_RUN = False

sys.path.insert(0, PROJECT + "/src")

from quantvesting import (
    Quantvesting, load_config, load_market_data, load_portfolio_data,
    create_run_id, PROSPECT_DISPLAY_COLUMNS, PORTFOLIO_DISPLAY_COLUMNS,
)

RUN_ID = create_run_id()

Mounted at /content/drive
Enter Portfolio ID (default: ankit): vinod


In [3]:
config = load_config(PROJECT + "/config/strategy.yaml")
qv = Quantvesting(config)

# Optional shared-market refresh; leave False unless a new Screener XLSX arrived.
REFRESH_SCREENER = False
if REFRESH_SCREENER:
    qv.ingest_screener(MARKET_DATA_DIR)

market_data = load_market_data(MARKET_DATA_DIR)
portfolio_data = load_portfolio_data(PORTFOLIO_DATA_DIR, portfolio_id=PORTFOLIO_ID)

### 2. Run the existing engine

The notebook only orchestrates the existing prospect/portfolio/decision layers.

In [4]:
df_prospects = qv.prospects(
    market_data, portfolio_data=portfolio_data, include_portfolio=True,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_portfolio, portfolio_summary = qv.portfolio(
    market_data, portfolio_data=portfolio_data, eod=EOD_RUN,
    portfolio_id=PORTFOLIO_ID, run_id=RUN_ID,
)

df_prospect_actions = qv.prospect_actions(df_prospects, top_n=10)
df_portfolio_actions = qv.portfolio_actions(df_portfolio)
df_rotation = qv.capital_rotation(df_prospects, df_portfolio)

### 3. Executive dashboard

The top section is designed to answer four questions quickly: **What do I own? What needs attention? Where is the remaining upside? Where is the next opportunity?**

In [5]:
qv.display_terminal(
     portfolio_data=portfolio_data,
    df_portfolio=df_portfolio,
    df_prospects=df_prospects,
    portfolio_summary=portfolio_summary,
    df_rotation=df_rotation,
    df_portfolio_actions=df_portfolio_actions,
    df_prospect_actions=df_prospect_actions,
    portfolio_id=PORTFOLIO_ID,
    run_id=RUN_ID,
    run_datetime=portfolio_summary.get("run_datetime"),
)

{'current': 542320.0,
 'deployed': 867500.0,
 'xirr': nan,
 'target_value': 933471.0,
 'target_profit': 391151.0,
 'target_profit_pct': 72.12549786104145,
 'portfolio_health_pct': 100.0,
 'portfolio_health_value': 542320.0,
 'core_allocation_pct': 100.0,
 'legacy_allocation_pct': 0.0,
 'out_of_universe_allocation_pct': nan,
 'out_of_universe_basis': 'unavailable',
 'top5_concentration_pct': 100.0,
 'top10_concentration_pct': 100.0,
 'top20_concentration_pct': 100.0,
 'pnl': -325180.0,
 'positions': 3,
 'prospects': 60,
 'weighted_remaining_upside_pct': 0.7212549786104145,
 'median_rrr': -0.47,
 'action_count': 0,
 'prospect_candidates': 0,
 'rotation_candidates': 0}

### 4. Interactive portfolio intelligence

Use the tables for drill-down. Sorting/filtering remains delegated to the existing Jupyter/Colab table adapter.

In [6]:
qv.display_dataframe(
    df_portfolio_actions,
    columns=[c for c in PORTFOLIO_DISPLAY_COLUMNS + ["Action", "ActionReason", "ActionEvidence"]
             if c in df_portfolio_actions.columns],
    sort_by="CurrAlloc%", ascending=False,
)

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,...,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category,Action,ActionReason,ActionEvidence
2,TCS,-0.69,-28.00,83.62,32.21,192660.0,-89600.0,230400.0,4230.71,-43.98,...,2.0,-0.47,42.48,16.85,X40,ATH,IT,HOLD,Thesis not yet at a configured review/target t...,Captured=-87% | Remaining upside=83.6%
1,INFY,-0.03,-22.07,74.51,36.00,126294.0,-48000.0,169500.0,1972.00,-40.86,...,3.0,-0.38,31.25,14.69,X40,NTT,IT,HOLD,Thesis not yet at a configured review/target t...,Captured=-61% | Remaining upside=74.5%
0,HDFCBANK,0.77,-56.84,50.68,-34.97,72178.0,-187580.0,142420.0,1073.00,-29.51,...,26.0,-2.60,26.26,1.61,X40,BTT,BANKS,HOLD,Thesis not yet at a configured review/target t...,Captured=N/A | Remaining upside=50.7%


In [7]:
qv.display_dataframe(
    df_prospect_actions,
    columns=[c for c in PROSPECT_DISPLAY_COLUMNS + ["Action", "Reason", "Evidence"]
             if c in df_prospect_actions.columns],
    sort_by="CumlRnk", ascending=True,
)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,...,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio,Action,Reason,Evidence
0,INFY,1972.00,-12.13,-40.86,13.67,X-LC,NC,49.0,47.92,74.51,...,14.69,1.0,2.1,X40,NTT,IT,MAIN,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=1
1,TCS,4230.71,-9.68,-43.98,11.08,X-LC,NC,48.0,28.30,83.62,...,16.85,2.0,3.1,X40,ATH,IT,MAIN,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=2
2,TMPV,600.00,-10.49,-84.91,7.09,X-LC,DC,36.0,9.38,92.62,...,6.04,3.0,54.0,XY24,NTT,AUTO,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=3
3,BSE,4391.73,4.08,-17.92,8.29,X-LC,NC,50.0,32.64,28.80,...,67.57,4.0,0.9,X40N,ATH,MISC,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=4
4,HINDUNILVR,2922.00,-10.71,-43.44,8.36,X-LC,NC,37.0,31.25,48.07,...,0.58,5.0,1.0,XY25,NTT,FMCG,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=5
5,ITC,452.00,-13.97,-31.98,11.93,X-LC,NC,40.0,14.93,71.15,...,3.37,6.0,1.8,X40,NTT,FMCG,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=6
6,HCLTECH,1853.42,-4.55,-15.93,6.77,X-LC,NC,47.0,32.99,43.30,...,26.34,7.0,1.2,X40,ATH,IT,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=7
7,HDFCBANK,1073.00,-14.00,-29.51,12.92,X-LC,NC,40.0,42.71,50.68,...,1.61,8.0,1.0,X40,BTT,BANKS,MAIN,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=8
8,M&M,3762.88,-4.06,-13.83,2.73,X-LC,DC,35.0,19.10,18.70,...,9.29,9.0,1.0,X40N,ATH,AUTO,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=9
9,VBL,669.42,-13.09,-32.47,9.08,X-LC,NC,36.0,22.22,64.23,...,7.00,10.0,0.4,X40N,ATH,FMCG,NA,BUY_CANDIDATE,High-ranked opportunity within the configured ...,CumlRnk=10


### 5. Visual views

In [8]:
qv.display_health_chart(df_portfolio)
qv.display_upside_chart(df_portfolio, top_n=12)
qv.display_prospect_opportunities(df_prospects, top_n=12)

### 6. Capital rotation review

This is advisory only. It does not issue an automatic sell instruction.

In [9]:
if df_rotation.empty:
    print("No capital-rotation candidates at the current configured thresholds.")
else:
    display(df_rotation)

No capital-rotation candidates at the current configured thresholds.


### 7. Run / date selector

Run manifests answer **which data/configuration produced a result**. EOD history provides the stored portfolio time series. Historical per-stock snapshots are not reconstructed by this selector because the current repository does not persist those per-stock snapshots.

In [10]:
qv.display_run_history_selector(portfolio_data, current_run_id=RUN_ID)

(Dropdown(description='Run:', options=('Latest', 'run_20260904_102125_e1ccc65f', 'run_20260904_102303_b5c54b4a', 'run_20260904_123651_1ec4273d', 'run_20260904_124019_87981f4a', 'run_20260904_124610_6bb8545a', 'run_20260904_134146_c1b49f3f', 'run_20260904_151102_0c8bd046', 'run_20260904_154250_f7bccd7d', 'run_20260904_154409_dea5c94b', 'run_20260904_154938_6797ef7c'), value='Latest'),
 Dropdown(description='EOD date:', options=('Latest',), value='Latest'))

### 8. HNI review checklist

1. Review the executive cards.
2. Read active actions and their evidence.
3. Inspect top remaining-upside holdings.
4. Inspect top prospect opportunities.
5. Review rotation candidates.
6. Check the selected run/date before sharing the report.

The investment engine remains unchanged; this notebook is the presentation layer.